# 04 — Google Meridian, as an independent comparison

pymc-marketing and Meridian are the two open media-mix implementations in current
use. They differ in more than syntax: Meridian is written for a geo-hierarchical
design, uses its own priors, and samples with TensorFlow Probability rather than
PyMC.

Running both on the same panel is a cross-implementation check in the same spirit
as the two CLV fits in notebook 06 — agreement between independently written
implementations is evidence, in a way a model agreeing with itself is not.

The gate in `metrics/env_gate.json` records whether Meridian could be imported and
exercised at all on this machine. A comparison that could not run is recorded as
such rather than quietly dropped.

In [ ]:
import json
import warnings

import pandas as pd

from athar import paths
from athar.provenance import read_metric

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

METRICS = paths.metrics_dir()
PROCESSED = paths.processed_dir()


def show(frame, caption=""):
    if caption:
        print(caption)
    print(frame.to_string(index=False))
    print()

In [ ]:
gate = read_metric("env_gate", METRICS)
print("platform:", gate["platform"])
print()
for name, result in gate["gates"].items():
    print(f"{name:18s} {result['status']}")
    if result["status"] == "fail":
        print("   ", result["error"])

In [ ]:
meridian_gate = gate["gates"]["meridian"]
if meridian_gate["status"] != "pass":
    print("Meridian did not pass its gate on this machine; the comparison below did not run.")
    print(meridian_gate.get("error"))
else:
    print("Meridian imported and exercised successfully:")
    print(json.dumps(meridian_gate["detail"], indent=2))

## Why the comparison is reported at the level of the gate

Meridian's data interface expects a geo-by-time array with its own coordinate
conventions and its own notion of media, reach and frequency inputs. Adapting the
panel to it faithfully is a piece of work in its own right, and adapting it
*unfaithfully* — flattening the geo dimension, guessing at the population scaling —
would produce a number that looks like a comparison and is not one.

What is claimed here is therefore narrow and true: Meridian installs and runs on
this machine, which was not obvious on Apple silicon with TensorFlow Probability,
and the version is recorded. What is **not** claimed is a fitted Meridian ROI for
this panel. An unfaithful adaptation would be worse than an absent one, and saying
so is more useful than a number nobody should trust.

In [ ]:
print("tensorflow:", gate["versions"].get("tensorflow"))
print("google-meridian:", gate["versions"].get("google-meridian"))
print("pymc-marketing:", gate["versions"].get("pymc-marketing"))